<a href="https://colab.research.google.com/github/GabrielMtzSoltero/MIEC/blob/main/Miec_limpiezaDeDatos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:

import numpy as np
import pandas as pd

# Módulos de Preprocesamiento de Scikit-Learn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score



In [15]:
# Generamos un dataset sintético que simula datos sucios/reales de clientes de un banco
data = {
    'Edad': [25, 48, np.nan, 33, 52, 23, 40, np.nan],            # Numérica continua con Nulos
    'Ingreso_Anual': [50000, 120000, 85000, np.nan, 140000, 32000, 95000, 60000], # Numérica con Nulos
    'Nivel_Educacion': ['Bachillerato', 'Maestria', 'Licenciatura', 'Licenciatura', 'Doctorado', 'Bachillerato', np.nan, 'Maestria'], # Categórica Ordinal
    'Ciudad': ['Madrid', 'Barcelona', 'Madrid', 'Valencia', 'Barcelona', 'Madrid', 'Valencia', 'Barcelona'], # Categórica Nominal
    'Monto_Credito': [12000, 45000, 30000, 18000, 55000, 8000, 35000, 22000] # Variable Objetivo (Y)
}

df = pd.DataFrame(data)

print("=== DATASET ORIGINAL (CON NULOS Y DIVERSOS TIPOS DE DATOS) ===")
print(df)
print("\nValores nulos por columna:")
print(df.isnull().sum())



=== DATASET ORIGINAL (CON NULOS Y DIVERSOS TIPOS DE DATOS) ===
   Edad  Ingreso_Anual Nivel_Educacion     Ciudad  Monto_Credito
0  25.0        50000.0    Bachillerato     Madrid          12000
1  48.0       120000.0        Maestria  Barcelona          45000
2   NaN        85000.0    Licenciatura     Madrid          30000
3  33.0            NaN    Licenciatura   Valencia          18000
4  52.0       140000.0       Doctorado  Barcelona          55000
5  23.0        32000.0    Bachillerato     Madrid           8000
6  40.0        95000.0             NaN   Valencia          35000
7   NaN        60000.0        Maestria  Barcelona          22000

Valores nulos por columna:
Edad               2
Ingreso_Anual      1
Nivel_Educacion    1
Ciudad             0
Monto_Credito      0
dtype: int64


In [16]:
X = df.drop(columns=['Monto_Credito'])
y = df['Monto_Credito']

# Clasificación de columnas por tipo de tratamiento
num_features = ['Edad', 'Ingreso_Anual']
ord_features = ['Nivel_Educacion']
nom_features = ['Ciudad']

# Orden jerárquico para la variable ordinal
educacion_orden = ['Bachillerato', 'Licenciatura', 'Maestria', 'Doctorado']



In [17]:

# A) Pipeline Numérico: Imputar faltantes con la Mediana + Escalado Estándar (StandardScaler)
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# B) Pipeline Categórico Ordinal: Imputar con el valor más frecuente + OrdinalEncoder
ord_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[educacion_orden]))
])

# C) Pipeline Categórico Nominal: Imputar + OneHotEncoder (Crear columnas dummy)
nom_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False)) # drop='first' evita la trampa de la multicolinealidad
])



In [18]:
# ==========================================
# 5. Integración Global con ColumnTransformer
# ==========================================
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_features),
    ('ord', ord_pipeline, ord_features),
    ('nom', nom_pipeline, nom_features)
])



In [19]:
# Aplicamos fit_transform sobre X
X_processed = preprocessor.fit_transform(X)

# Recuperamos los nombres de las columnas transformadas
nom_encoded_cols = preprocessor.named_transformers_['nom']['encoder'].get_feature_names_out(nom_features)
all_column_names = num_features + ord_features + list(nom_encoded_cols)

# Convertimos a DataFrame para visualizar limpiamente el resultado
df_processed = pd.DataFrame(X_processed, columns=all_column_names)

print("=== DATASET PREPROCESADO Y LISTO PARA MACHINE LEARNING ===")
print(df_processed)
print("-" * 60)

# Unimos el preprocesador y el modelo de Scikit-Learn en un solo flujo
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# Entrenamos el pipeline completo
full_pipeline.fit(X, y)

# Predicción de ejemplo
y_pred = full_pipeline.predict(X)

print("=== PREDICCIONES DEL MODELO FIT-TRANSFORM ===")
for i, pred in enumerate(y_pred):
    print(f"Cliente {i+1}: Predicción Credito = ${pred:,.2f} | Real = ${y.iloc[i]:,.2f}")

=== DATASET PREPROCESADO Y LISTO PARA MACHINE LEARNING ===
       Edad  Ingreso_Anual  Nivel_Educacion  Ciudad_Madrid  Ciudad_Valencia
0 -1.246373      -0.995721              0.0            1.0              0.0
1  1.193336       1.092683              2.0            0.0              0.0
2 -0.026519       0.048481              1.0            1.0              0.0
3 -0.397779       0.048481              1.0            0.0              1.0
4  1.617633       1.689370              3.0            0.0              0.0
5 -1.458522      -1.532739              0.0            1.0              0.0
6  0.344741       0.346824              0.0            0.0              1.0
7 -0.026519      -0.697378              2.0            0.0              0.0
------------------------------------------------------------
=== PREDICCIONES DEL MODELO FIT-TRANSFORM ===
Cliente 1: Predicción Credito = $11,828.25 | Real = $12,000.00
Cliente 2: Predicción Credito = $46,988.86 | Real = $45,000.00
Cliente 3: Predicción Cr